# S6_LOC_cmv_predict.ipynb
## Hybrid PV-Forecast with Cloud Motion Vector (OpenCV Optical Flow) and Catboost-Models
Conda Env: scikit-learn
last edit: 13.09.2026

Copyright (c) 2026 Andreas Mätzler

All rights reserved.

You may not use, copy, modify, distribute, or reproduce this code for any purpose without explicit written permission from the author.

In [ ]:
import numpy as np
import pandas as pd
import cv2
import os
from osgeo import gdal
gdal.UseExceptions()
from datetime import datetime, timedelta
import datetime
import shlex
import catboost
from catboost import *
import sklearn.metrics as metrics
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.metrics import mean_absolute_percentage_error

import rasterio
import plotly.express as px
import plotly.graph_objects as go
import time

# load feature sets from common module
import sys, os; sys.path.insert(0, os.path.abspath('./common'))
from feature_sets import FEATURE_SETS, SENSOR

### set variables

In [ ]:
# dictionary with 7 different scenarios for testing of the prediction model
PREDICTION_SCENARIOS = [
    {"timestamp": datetime.datetime.strptime("29.01.2018 07:35:11", "%d.%m.%Y %H:%M:%S"),"description": "SC1 29.01.2018 07:30 Clear winter morning with shadows"},
    {"timestamp": datetime.datetime.strptime("22.03.2018 14:33:53", "%d.%m.%Y %H:%M:%S"),"description": "SC2 22.03.2018 14:30 Cloudy evening"},
    {"timestamp": datetime.datetime.strptime("11.08.2018 07:33:53", "%d.%m.%Y %H:%M:%S"),"description": "SC3 11.08.2018 07:30 Cloudy morning"},
    {"timestamp": datetime.datetime.strptime("12.08.2018 12:13:53", "%d.%m.%Y %H:%M:%S"),"description": "SC4 12.08.2018 12:00 Sunny Summerday"},
    {"timestamp": datetime.datetime.strptime("14.08.2018 12:13:53", "%d.%m.%Y %H:%M:%S"),"description": "SC5 14.08.2018 12:00 Rainstorm from north west"},
    {"timestamp": datetime.datetime.strptime("14.08.2018 12:37:23", "%d.%m.%Y %H:%M:%S"),"description": "SC6 14.08.2018 12:30 Rainstorm from north west (0.5h later)"},
    {"timestamp": datetime.datetime.strptime("14.11.2018 09:07:51", "%d.%m.%Y %H:%M:%S"),"description": "SC7 14.11.2018 09:00 Autumn day with fog in lower regions"},
    {"timestamp": datetime.datetime.strptime("16.12.2018 11:02:21", "%d.%m.%Y %H:%M:%S"),"description": "SC8 16.12.2018 11:00 Winter day with snow and clouds"},
]

# dataset selection, smaller datasets for testing
csv_dataset = "alle_151_Netzeinspeiser"
#csv_dataset = "Radius_5000m"
#csv_dataset = "Radius_200m"

# set path variables
eumetsat_path = "C:\\Users\\Andreas\\Documents\\UNIGIS\\2017\\Master-Thesis\\Daten\\Satellite\\EUMETSAT"
eumetsat_geotiff_path = eumetsat_path + "\\Result_Timestamped\\GeoTIFF\\Extended_Clip" # path to Extended clipped EUMETSAT-GeoTIFF images
eumetsat_forecast_output_path = eumetsat_path + "\\Forecast" # patch to save forecast geotiff
result_path = r'.\results' # path for result diagrams/CSVs
models_path = r'.\models' # path for CatBoost models
dataframe_path = '../../Daten/Solar_Load_Profile/2018_Features_Solar_Load_Profile_' + csv_dataset + '.csv'

# set list with all SEVIRI bands and correct order
# source: https://eumetsat.int/0-degree-service
all_bands = ['HRV','VIS006','VIS008','IR_016','IR_039','WV_062','WV_073','IR_087','IR_097','IR_108','IR_120','IR_134']
#all_bands = ["HRV"]

# set forecast range in step in minutes
forecast_range = 180
forecast_step = 15

map feature sets to their trained CBM model files

In [3]:

# derive cbm path from fs_name
for fs_name in FEATURE_SETS:
    FEATURE_SETS[fs_name]['cbm'] = os.path.join(models_path, f"model_{fs_name}.cbm")

# load all models once into a dictionary for later use
loaded_models = {}
for fs_name, fs_conf in FEATURE_SETS.items():
    m = CatBoostRegressor()
    m.load_model(fs_conf['cbm'])
    loaded_models[fs_name] = m
    print("loaded model for", fs_name, fs_conf['cbm'])

loaded model for FS1_TIME .\models\model_FS1_TIME.cbm
loaded model for FS2_GIS_TIME .\models\model_FS2_GIS_TIME.cbm
loaded model for FS3_GIS_SENSOR_TIME .\models\model_FS3_GIS_SENSOR_TIME.cbm
loaded model for FS4_SEV4_TIME .\models\model_FS4_SEV4_TIME.cbm
loaded model for FS5_SEV4_GIS_TIME .\models\model_FS5_SEV4_GIS_TIME.cbm
loaded model for FS6_SEV4_GIS_SENSOR_TIME .\models\model_FS6_SEV4_GIS_SENSOR_TIME.cbm
loaded model for FS7_SEV12_TIME .\models\model_FS7_SEV12_TIME.cbm
loaded model for FS8_SEV12_GIS_TIME .\models\model_FS8_SEV12_GIS_TIME.cbm
loaded model for FS9_SEV12_GIS_SENSOR_TIME .\models\model_FS9_SEV12_GIS_SENSOR_TIME.cbm


### set date and time for last and next to last Image  

In [4]:
# Function: to round actual/predict time to Quarter
def compute_rounded_times(date_predict):
    # date_last_image_rounded = date_predict - (date_predict - date_predict.min) % timedelta(minutes=15) # not safe method
    date_last_image_rounded = date_predict - timedelta(minutes=date_predict.minute % 15,seconds=date_predict.second,microseconds=date_predict.microsecond)
    date_last_image=date_last_image_rounded.strftime("%Y-%m-%d %H_%M_%S")
    print("actual/predict datetime:             {}".format(date_predict)) # pring date an time
    print("rounded datetime last image:         {}".format(date_last_image))  # printed in default formatting
    # substract 15min from date for next to last image
    date_next_to_last_rounded = date_last_image_rounded - datetime.timedelta(minutes=15)
    date_next_to_last_image=date_next_to_last_rounded.strftime("%Y-%m-%d %H_%M_%S")
    print("rounded datetime next to last image: {}".format(date_next_to_last_image))  # printed in default formatting
    return date_last_image_rounded, date_last_image, date_next_to_last_image

# for testing the function
# date_predict = datetime.datetime.strptime("29.01.2018 07:35:11", "%d.%m.%Y %H:%M:%S")
# compute_rounded_times(date_predict)

In [5]:
# Function: load all images and bands from GeoTIFF into a nested dictionary
def load_images_and_bands(date_last_image, date_next_to_last_image, eumetsat_geotiff_path,all_bands):
    images=["next_to_last_image","last_image"]

    # create empty nested dictionary for images and bands
    images_data = {}
    for image in images:
        images_data[image] = {}

    # loop for next to last and last images
    for image in images:
        if image == "next_to_last_image":
            date_processing = date_next_to_last_image
        else:
            date_processing = date_last_image

        datasets = ["IR_VIS_WR","HRV"]
                
        # loop trough all datasets
        for dataset in datasets:
            # set bands for each dataset
            if dataset == "HRV":
                bands = ["HRV"]
            else:
                bands = all_bands[1:] # exclude HRV from all_bands list for IR_VIS_WR dataset
                
            # set starting band number for iteration
            band_nr = 1

            # loop through all bands of the given dataset
            for band in bands:
                
                # set path and filename dynamicly
                geotiff_filename = os.path.join(eumetsat_geotiff_path, date_processing + "_{}.tif".format(dataset))        

                # print for debugging
                # print("### processing file:",geotiff_filename,band,band_nr)
                
                # read geotiff-images
                image_processing = gdal.Open(geotiff_filename)
                
                # create numpy arrays with dynamic names from variable image and band e.g. "next_to_last_IR_039"
                dynamic_array = np.array(image_processing.GetRasterBand(band_nr).ReadAsArray().astype(np.float32) )
                dynamic_array_name = image + "_"+ band
                # globals()[dynamic_array_name] = dynamic_array
                images_data[image][band] = dynamic_array

                # increase the band number
                band_nr = band_nr + 1

    # get gdal-parameters from last processing image for writing results as geotiff
    gdal_params = {
        "data_type" : image_processing.GetRasterBand(1).DataType,
        "geotransform" : image_processing.GetGeoTransform(),
        "spatialreference" : image_processing.GetProjection(),
        "ncol" : image_processing.RasterXSize,
        "nrow" : image_processing.RasterYSize,
        "nband" : 1
    }

    return images_data, gdal_params

# for testing the function
# images_data, gdal_params = load_images_and_bands(
#     date_last_image, date_next_to_last_image, eumetsat_geotiff_path
# )

# print(array_next_to_last_vis006.shape, array_next_to_last_vis006.dtype)

### Detect motion flow from HRV-Files and predict on all bands

In [6]:
# Function: export_geotiff for export result as single-band GeoTIFF-Raster
def export_geotiff(path, file, band, ncol, nrow, nband, data_type, geotransform, spatialreference, image):
    # create output folder for geotiff
    if not os.path.exists(path + "\\" + band):
        os.makedirs(path + "\\" + band)

    # set geotiff filename
    output_geotiff_file = os.path.join(path + "\\" + band, file + ".tif")
    
    # create geotiff file of forecast
    # Source: https://borealperspectives.org/2014/01/16/data-type-mapping-when-using-pythongdal-to-write-numpy-arrays-to-geotiff/
    driver = gdal.GetDriverByName("GTiff")
    # Source: https://kokoalberti.com/articles/geotiff-compression-optimization-guide/
    #dst_dataset = driver.Create(output_geotiff_file, ncol, nrow, nband, data_type,  [ 'COMPRESS=ZSTD', 'PREDICTOR=3', 'TILED=YES', 'NUM_THREADS=ALL_CPUS' ])
    dst_dataset = driver.Create(output_geotiff_file, ncol, nrow, nband, data_type,  [ 'COMPRESS=PACKBITS', 'TILED=YES', 'NUM_THREADS=ALL_CPUS' ])
    dst_dataset.SetGeoTransform(geotransform)
    dst_dataset.SetProjection(spatialreference)
    dst_dataset.GetRasterBand(1).SetDescription(band)
    dst_dataset.GetRasterBand(1).WriteArray(image)
    dst_dataset = None

In [7]:
# Function: print array min/max/median/mean for debugging
def print_array(name, array):
    print("  --{} min:    {}".format(name,str(np.min(array))))
    print("    {} max:    {}".format(name,str(np.max(array))))
    print("    {} median: {}".format(name,str(np.median(array))))
    print("    {} mean:   {}".format(name,str(np.mean(array))))

In [8]:
# Function: array2raster2 for export a array as Single Band-Geotiff Raster
def array2raster2(path, fname, matriz, geot, proj):
    # create output folder for geotiff
    if not os.path.exists(path):
        os.makedirs(path)

    # Source: https://gis.stackexchange.com/questions/189942/writing-3-channels-to-8-bit-tif-in-python-using-gdal
    drv = gdal.GetDriverByName("GTiff")
    dst_ds = drv.Create(os.path.join(path + "\\", fname), matriz.shape[1], matriz.shape[0], 3, gdal.GDT_Byte)
    dst_ds.SetGeoTransform(geot)
    dst_ds.SetProjection(proj)
    dst_ds.GetRasterBand(3).WriteArray(matriz[:, :, 0])  
    dst_ds.GetRasterBand(2).WriteArray(matriz[:, :, 1])  
    dst_ds.GetRasterBand(1).WriteArray(matriz[:, :, 2])
    dst_ds.FlushCache()
    dst_ds=None

In [9]:
# Function: processOptical Flow + Export results as GeoTIFF-Raster
def run_optical_flow_and_export(images_data, gdal_params, date_last_image, forecast_range, forecast_step,eumetsat_forecast_output_path,all_bands):

    # get gdal parameters for writing results as geotiff
    data_type = gdal_params["data_type"]
    geotransform = gdal_params["geotransform"]
    spatialreference = gdal_params["spatialreference"]
    ncol = gdal_params["ncol"]
    nrow = gdal_params["nrow"]
    nband = gdal_params["nband"]

    ### create and normalize arrays to 8 bit (0-255) for CMV with OpenCV

    # create empty nested dictionary for scaled data
    scaled_data = {}

    for label in ["next_to_last_image", "last_image"]:
        scaled_data[label] = {}

    # loop trough all bands
    for band in all_bands:
        print(f"### processing values: next_to_last_{band} last_image_{band}")
        next_to_last_image_processing_array = images_data["next_to_last_image"][band]
        last_image_processing_array = images_data["last_image"][band]

        # concatenate arrays of each band for getting min/max values for scaling
        concatenated_array = np.concatenate((next_to_last_image_processing_array, last_image_processing_array), axis=0)

        # normalize arrays to concatenated max scale 255 within 0 and 255
        next_to_last_image_scaled = np.uint8((next_to_last_image_processing_array) / (np.max(concatenated_array)) * 255)
        last_image_scaled = np.uint8((last_image_processing_array) / (np.max(concatenated_array)) * 255)

        # print min/max values for scale debugging
        # print_array(f"next_to_last_{band}_processing_array", next_to_last_image_processing_array)
        # print_array(f"last_image_{band}_processing_array", last_image_processing_array)
        # print_array(f"{band}_concatenated_array", concatenated_array)
        # print_array(f"next_to_last_{band}_scaled", next_to_last_image_scaled)
        # print_array(f"last_image_{band}_scaled", last_image_scaled)

        # Ergebnisse im Dictionary statt in globals() ablegen
        scaled_data["next_to_last_image"][band] = next_to_last_image_scaled
        scaled_data["last_image"][band] = last_image_scaled

        # export the two base images (next to last and last) as geotiff raster
        export_geotiff(eumetsat_forecast_output_path, date_last_image + " -15min_" + band, band, ncol, nrow, nband, data_type, geotransform, spatialreference,next_to_last_image_processing_array)
        export_geotiff(eumetsat_forecast_output_path, date_last_image + " +0min_" + band, band,ncol, nrow, nband, data_type, geotransform, spatialreference,last_image_processing_array)

    return scaled_data

In [10]:
# Function: visualize Flow-Motion from HRV-Band with arrows
def draw_flow(img, flow, step=16):
    h, w = img.shape[:2]
    y, x = np.mgrid[step/2:h:step, step/2:w:step].reshape(2,-1).astype(int)
    fx, fy = flow[y,x].T*5
    lines = np.vstack([x, y, x+fx, y+fy]).T.reshape(-1, 2, 2)
    lines = np.int32(lines + 0.5)
    vis = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    #cv2.polylines(vis, lines, 0, (0, 0, 255))
    for (x1, y1), (_x2, _y2) in lines:
        # cv2.circle(vis, (x1, y1), 1, (0, 0, 255), -1)
        cv2.arrowedLine(vis, (x1, y1), (_x2, _y2), (0, 255, 0), (1), 1, 0, 0.5 )
    return vis

In [ ]:
# Function: compute Optical Flow forecast for all steps and export as GeoTIFF
def compute_optical_flow_forecast(images_data,scaled_data,gdal_params,date_last_image,forecast_range,forecast_step,eumetsat_forecast_output_path,all_bands):

    # get gdal parameters for writing results as geotiff
    data_type = gdal_params["data_type"]
    geotransform = gdal_params["geotransform"]
    spatialreference = gdal_params["spatialreference"]
    ncol = gdal_params["ncol"]
    nrow = gdal_params["nrow"]
    nband = gdal_params["nband"]

    # create a nested dictionary for scaled and array data
    state = {
        "scaled": {"next_to_last_image": dict(scaled_data["next_to_last_image"]),"last_image": dict(scaled_data["last_image"]),},
        "array": {"next_to_last_image": dict(images_data["next_to_last_image"]),"last_image": dict(images_data["last_image"]), }
    }

    forecast_results = {}  # create empty dictionary for forecast results

    # Optical-Flow-Algorithmus (Parameter nach Urbich et al. 2018)
    # Parameters from "A Novel Approach for the Short-Term Forecast of the Effective Cloud Albedo" by Isabel Urbich et al. 2018
            # https://www.mdpi.com/2072-4292/10/6/955/htm#table_body_display_remotesensing-10-00955-t002
            #optical_flow = cv2.optflow.DualTVL1OpticalFlow_create(0.1, 0.03, 0.3, 3, 3, 0.01, 10, 2, 0.5, 0.1, 5, 0 )
    
            # Documentation:  DualTVL1OpticalFlow_create([, tau [, lambda[, theta[, nscales[, warps[, epsilon[, innnerIterations[, outerIterations[, scaleStep[, gamma[, medianFiltering[, useInitialFlow]]]]]]]]]]]])
            # Default Values: DualTVL1OpticalFlow_create([, 0.25[, 0.15  [, 0.3  [, 5      [, 5    [, 0.01   [, 30              [, 10             [, 0.8      [, 0.0  [, 5              [, false         ]]]]]]]]]]]])
            # recom. Values:  DualTVL1OpticalFlow_create([, 0.1 [, 0.03  [, 0.3  [, 3      [, 3    [, 0.01   [, 10              [, 2              [, 0.5      [, 0.1  [, 5              [, false         ]]]]]]]]]]]])
    
     
    # optimized parameters for better results with resampled EUMETSAT images
    opticalflow = cv2.optflow.DualTVL1OpticalFlow_create(
            tau=0.1,
            lambda_=0.003,       # higher for finer cloud structures
            theta=0.3,
            nscales=6,          
            warps=6,            
            epsilon=0.005,
            innnerIterations=30, # typo detected in 'innerIterations'
            outerIterations=2,
            scaleStep=0.5,
            gamma=0.2,          # maintained for brightness changes
            medianFiltering=5,
            useInitialFlow=False
        )

    # optimized parameters for better results with resampled EUMETSAT images
    # opticalflow = cv2.optflow.DualTVL1OpticalFlow_create(
    #         tau=0.1,
    #         lambda_=0.005,       # higher for finer cloud structures
    #         theta=0.3,
    #         nscales=5,          
    #         warps=6,            
    #         epsilon=0.005,
    #         innnerIterations=30, # typo detected in 'innerIterations'
    #         outerIterations=2,
    #         scaleStep=0.8,
    #         gamma=0.2,          # maintained for brightness changes
    #         medianFiltering=5,
    #         useInitialFlow=False
    #     )

    # Use Hue, Saturation, Value colour model q
    hsv = np.zeros([nrow,ncol,3], dtype=np.uint8)
    hsv[..., 1] = 255

    # Set counter
    counter = forecast_step

    # loop trough forecast range with stepsize in minutes - step=15min and range=180min
    while counter <= forecast_range:
        # set forecast name variable
        forecast_name = date_last_image + " +" + str(counter) + "min"
        print(f"### processing Flow {forecast_name}")

        # calculate optical flow between next to last and last image as array from HRV because it has the highest resolution
        next_to_last_hrv_scaled = state["scaled"]["next_to_last_image"]["HRV"]
        last_hrv_scaled = state["scaled"]["last_image"]["HRV"]

        flow = opticalflow.calc(next_to_last_hrv_scaled, last_hrv_scaled, None)
        print_array("flow", flow)
        
        # optical_flow = cv2.optflow.DualTVL1OpticalFlow_create(0.1, 0.003, 0.3, 6, 6, 0.005, 30, 2, 0.5, 0.2, 5, 0 )

        # Optical Flow with Farnebäck method (not used in final version)
        # calculate optical flow between next to last and last image as array with farnebäck method
        # Farnebäck, G. (2003, June). Two-frame motion estimation based on polynomial expansion. In Scandinavian conference on Image analysis (pp. 363-370). Springer, Berlin, Heidelberg.
        # flow = cv2.calcOpticalFlowFarneback(next_to_last_image_HRV_scaled, last_image_HRV_scaled, None, 0.5, 3, 15, 3, 5, 1.2, 0)

        # transform Flow-Field for cv2.remap() backward mapping
        h, w = flow.shape[:2]
        flow_transformed = -flow
        flow_transformed[..., 0] += np.arange(w)
        flow_transformed[..., 1] += np.arange(h)[:, np.newaxis]

        forecast_results[counter] = {}
      
        # print flow values for debugging
        print("### processing Flow:",forecast_name)
        print_array("flow",flow)

        # convert flow array in color image to visualize the flow
        mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])
        hsv[..., 0] = ang * 180 / np.pi / 2
        hsv[..., 2] = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX)
        colored_flow = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)   
    
        # transform flow array for generating a new frame
        # Snippet from https://github.com/opencv/opencv/issues/11068
        h, w = flow.shape[:2]
        flow_transformed = -flow
        flow_transformed[:,:,0] += np.arange(w)
        flow_transformed[:,:,1] += np.arange(h)[:,np.newaxis]

        forecast_results[counter] = {}
        
        # use calculated flow to loop trough all bands to forecast
        for band in all_bands:
            last_scaled = state["scaled"]["last_image"][band]
            last_array = state["array"]["last_image"][band]

            # Forecast the next frame using the flow field and remap the last image
            forecast_scaled = cv2.remap(last_scaled, flow_transformed, None,interpolation=cv2.INTER_LINEAR,borderMode=cv2.BORDER_CONSTANT, borderValue=255)

            # print min/max values for scale debugging
            if band =="HRV":
                print_array("forecast_scaled",forecast_scaled)

                # # show colored flow in window
                # window_name_colored_flow = "Colored flow " + forecast_name
                # cv2.namedWindow(window_name_colored_flow, cv2.WINDOW_NORMAL)
                # cv2.resizeWindow(window_name_colored_flow, 900, 900)
                # cv2.imshow(window_name_colored_flow, colored_flow)
                
                print("### processing flow arrow:")
                flow_arrow = draw_flow(forecast_scaled, flow, 25)
                
                # # show flow as arrows with last HRV-image in windows
                # window_name_flow_as_arrow = "Flow as Arrow " + forecast_name
                # cv2.namedWindow(window_name_flow_as_arrow, cv2.WINDOW_NORMAL)
                # cv2.resizeWindow(window_name_flow_as_arrow, 900, 900)
                # cv2.imshow(window_name_flow_as_arrow,flow_arrow)

                # export forecast flow as geotiff file
                array2raster2(eumetsat_forecast_output_path + "\\FLOW", forecast_name + "_FLOW.tif", colored_flow, geotransform, spatialreference)
                
                # export forecast HRV with Arrows as geotiff file
                array2raster2(eumetsat_forecast_output_path + "\\HRV_ARROWS", forecast_name + "_HRV_ARROWS.tif", flow_arrow, geotransform, spatialreference)

            # scale value from 255 (greyscale) to max value from last image
            forecast_array = (forecast_scaled.astype(np.float32) / 255) * np.max(last_array.astype(float))
            
            if band == "HRV":
                print_array("forecast_scaled", forecast_scaled)
                print_array("forecast_array", forecast_array)

            # export forecast as geotiff file
            export_geotiff(eumetsat_forecast_output_path,forecast_name + "_" + band,band,ncol,nrow,nband,data_type,geotransform,spatialreference,forecast_array)

            # get pixel value for debugging https://gis.stackexchange.com/questions/118397/storing-result-from-gdallocationinfo-as-variable-in-python
            # lat=47.487297
            # lon=9.737721
            # result = os.popen('C:/Progra~1/QGIS3~1.10/bin/gdallocationinfo -valonly -wgs84 "' + os.path.join(eumetsat_forecast_output_path + "\\" + band, forecast_band_name) + '.tif" ' + str(lon) + ' ' + str(lat)).read()
            # print("    Prognose: ", forecast_band_name, result)

            # Switch - Make last image to next to last frame for each band (next_to_last <- last, last <- forecast)
            state["scaled"]["next_to_last_image"][band] = last_scaled.copy()
            state["scaled"]["last_image"][band] = forecast_scaled.copy()
            state["array"]["next_to_last_image"][band] = last_array.copy()
            state["array"]["last_image"][band] = forecast_array.copy()

        # Increment counter with forecast_step before next frame
        counter += forecast_step
        cv2.waitKey(0)

    cv2.destroyAllWindows()

In [12]:
# loop through all prediction scenarios

all_scenario_results = []

for scenario in PREDICTION_SCENARIOS:
    date_predict = scenario["timestamp"]
    print(f"#### Scenario: {scenario['description']} ({date_predict})")

    date_last_image_rounded, date_last_image, date_next_to_last_image = compute_rounded_times(date_predict)

    images_data, gdal_params = load_images_and_bands(date_last_image,date_next_to_last_image,eumetsat_geotiff_path,all_bands)

    scaled_data = run_optical_flow_and_export(images_data,gdal_params,date_last_image,forecast_range,forecast_step,eumetsat_forecast_output_path,all_bands)

    forecast_results = compute_optical_flow_forecast(images_data,scaled_data,gdal_params,date_last_image,forecast_range,forecast_step,eumetsat_forecast_output_path,all_bands)

    all_scenario_results.append({
        "scenario": scenario["description"],
        "timestamp": date_predict,
        "forecast_results": forecast_results
    })

#### Scenario: SC1 29.01.2018 07:00 Clear winter morning with shadows (2018-01-29 07:35:11)
actual/predict datetime:             2018-01-29 07:35:11
rounded datetime last image:         2018-01-29 07_30_00
rounded datetime next to last image: 2018-01-29 07_15_00
### processing values: next_to_last_HRV last_image_HRV
### processing values: next_to_last_VIS006 last_image_VIS006
### processing values: next_to_last_VIS008 last_image_VIS008
### processing values: next_to_last_IR_016 last_image_IR_016
### processing values: next_to_last_IR_039 last_image_IR_039
### processing values: next_to_last_WV_062 last_image_WV_062
### processing values: next_to_last_WV_073 last_image_WV_073
### processing values: next_to_last_IR_087 last_image_IR_087
### processing values: next_to_last_IR_097 last_image_IR_097
### processing values: next_to_last_IR_108 last_image_IR_108
### processing values: next_to_last_IR_120 last_image_IR_120
### processing values: next_to_last_IR_134 last_image_IR_134
### process

In [13]:
def regression_results(y_true, y_pred):
    results = {}  # create empty dictionary to store results
    
    y_true = y_true.reset_index(drop=True)
    y_pred = y_pred.reset_index(drop=True)

    n = len(y_true)
    results['N Samples'] = n

    # R2 undefined for less than two samples
    results['r2'] = np.nan if n < 2 else r2_score(y_true, y_pred)

    results['explained Variance'] = metrics.explained_variance_score(y_true, y_pred)
    results['MAE'] = metrics.mean_absolute_error(y_true, y_pred)
    results['MSE'] = metrics.mean_squared_error(y_true, y_pred)
    results['RMSE'] = np.sqrt(results['MSE'])
    results['median AE'] = metrics.median_absolute_error(y_true, y_pred)
    results['MAPE'] = mean_absolute_percentage_error(y_true, y_pred)

    errors = y_true - y_pred
    results['Bias'] = errors.mean()  # vectorized, avoids per-row loop

    # MAAPE, bounded alternative to MAPE for near-zero values
    EPSILON = 1e-10
    results['MAAPE'] = np.mean(np.arctan(np.abs(errors / (y_true + EPSILON))))

    # print results
    print("--Metrics:")
    print('  explained_variance: ', round(results['explained Variance'], 4))
    if not np.isnan(results['r2']):
        print('  R²: ', round(results['r2'], 4))
    # print('  MAE: ', round(results['MAE'], 4))
    print('  MSE: ', round(results['MSE'], 4))
    print('  RMSE: ', round(results['RMSE'], 4))
    # print('  MAPE: ', round(results['MAPE'], 4))
    print('  MAAPE: ', round(results['MAAPE'], 4))
    # print('  Bias: ', round(results['Bias'], 4))
    print("Dataframe SHAPE: ", y_true.shape, y_pred.shape)

    return results

loop over all Feature-Sets and matching CBM-models for CatBoost prediction

In [14]:
dataframe = pd.read_csv(dataframe_path,parse_dates=['TIMESTAMP'])

# time-based cyclic features
dataframe['HOUR_DEZ'] = (pd.to_datetime(dataframe['TIMESTAMP']).dt.hour + pd.to_datetime(dataframe['TIMESTAMP']).dt.minute / 60)
dataframe['DAY_YEAR'] = pd.to_datetime(dataframe['TIMESTAMP']).dt.dayofyear
dataframe['SIN_HOUR'] = np.sin(2 * np.pi * dataframe['HOUR_DEZ'] / 24)
dataframe['COS_HOUR'] = np.cos(2 * np.pi * dataframe['HOUR_DEZ'] / 24)
dataframe['SIN_DAY'] = np.sin(2 * np.pi * dataframe['DAY_YEAR'] / 365)
dataframe['COS_DAY'] = np.cos(2 * np.pi * dataframe['DAY_YEAR'] / 365)

# sort like in training notebook
dataframe.sort_values(by=['TIMESTAMP', 'NEI_ID'], inplace=True)

print(dataframe.shape)

(2269939, 40)


main loop: scenario / feature-set / leadtime / NON SEVIRI or SEVIRI

In [15]:
# Initialize collection lists for predictions and metrics
all_results = []
metrics_results = []

for scenario_result in all_scenario_results:

    scenario_name = scenario_result["scenario"]
    date_predict = scenario_result["timestamp"]
    forecast_results = scenario_result["forecast_results"]

    # Compute rounded timestamp bounds for image loading
    date_last_image_rounded, date_last_image, date_next_to_last_image = compute_rounded_times(date_predict)

    print(f"#### Scenario {scenario_name} ({date_predict})")

    for fs_name, fs_conf in FEATURE_SETS.items():

        fs_columns = fs_conf['features']
        model = loaded_models[fs_name]

        if not isinstance(fs_columns, list):
            raise TypeError(f"FEATURE_SETS['{fs_name}']['features'] is a {type(fs_columns)}, expected a list!")

        print("### using model:", fs_name)

        counter = 0
        dataframe_result = pd.DataFrame()
        skipped_steps = 0
        sensor_lt0_snapshot = None

        while counter <= forecast_range:

            forecast_name = date_last_image + " +" + str(counter) + "min"
            date_prediction = date_last_image_rounded + datetime.timedelta(minutes=counter)

            # Filter dataframe for current prediction timestamp
            timefiltered_dataframe = dataframe[dataframe['TIMESTAMP'] == date_prediction].copy()

            if timefiltered_dataframe.empty:
                print("no rows for", date_prediction, "-> skip")
                skipped_steps += 1
                counter += forecast_step
                continue

            # Handle sensor feature persistence from leadtime 0
            if all(col in fs_columns for col in SENSOR):
                if counter == 0:
                    sensor_lt0_snapshot = timefiltered_dataframe[['NEI_ID'] + SENSOR].copy()
                elif sensor_lt0_snapshot is not None:
                    timefiltered_dataframe = (
                        timefiltered_dataframe.drop(columns=SENSOR)
                        .merge(sensor_lt0_snapshot, on='NEI_ID', how='left')
                    )

            # Extract spatial coordinates for raster sampling
            coords = [(x, y) for x, y in zip(timefiltered_dataframe.LON_WGS84, timefiltered_dataframe.LAT_WGS84)]

            bands_needed = [b for b in all_bands if b in fs_columns]
            has_seviri = len(bands_needed) > 0

            # Sample CMV forecasted rasters and preserve original values
            for band in bands_needed:
                forecast_band_name = forecast_name + "_" + band
                forecast_file = os.path.join(eumetsat_forecast_output_path + "\\" + band, forecast_band_name + ".tif")

                try:
                    forecast_raster = rasterio.open(forecast_file)
                except Exception as e:
                    print("could not open File!", forecast_file,e)
                    continue

                if band in timefiltered_dataframe.columns:
                    timefiltered_dataframe.rename(columns={band: band+"_ORG"}, inplace=True)

                timefiltered_dataframe[band] = [x[0] for x in forecast_raster.sample(coords)]

            # Path 1: CatBoost Prediction using CMV extrapolated features
            features_cmv = timefiltered_dataframe[list(fs_columns)].copy()
            y_pred_cmv = model.predict(features_cmv, task_type="GPU")

            # Path 2: CatBoostPrediction using ground truth SEVIRI features
            if has_seviri:
                # Map original band columns for reference prediction
                org_cols_map = {col: col + "_ORG" if col in bands_needed else col for col in fs_columns}
                features_org = timefiltered_dataframe[[org_cols_map[col] for col in fs_columns]].copy()
                features_org.columns = list(fs_columns) # set column names back to original for model input
                y_pred_org = model.predict(features_org, task_type="GPU")
            else:
                y_pred_org = y_pred_cmv.copy()

            # Assign prediction results to dataframe
            timefiltered_dataframe['SPECIFIC_YIELD_PREDICTION'] = y_pred_cmv
            timefiltered_dataframe['SPECIFIC_YIELD_PRED_CMV'] = y_pred_cmv
            timefiltered_dataframe['SPECIFIC_YIELD_PRED_ORG'] = y_pred_org

            # Calculate error components for hybrid analysis
            timefiltered_dataframe['SPECIFIC_YIELD_ERROR'] = timefiltered_dataframe['SPECIFIC_YIELD_PRED_CMV'] - timefiltered_dataframe['SPECIFIC_YIELD']
            timefiltered_dataframe['ERROR_TOTAL'] = timefiltered_dataframe['SPECIFIC_YIELD_PRED_CMV'] - timefiltered_dataframe['SPECIFIC_YIELD']
            timefiltered_dataframe['ERROR_REGRESSION'] = timefiltered_dataframe['SPECIFIC_YIELD_PRED_ORG'] - timefiltered_dataframe['SPECIFIC_YIELD']
            timefiltered_dataframe['ERROR_CMV'] = timefiltered_dataframe['SPECIFIC_YIELD_PRED_CMV'] - timefiltered_dataframe['SPECIFIC_YIELD_PRED_ORG']

            # Metadata columns for aggregation and filtering
            timefiltered_dataframe['SCENARIO'] = scenario_name
            timefiltered_dataframe['FEATURE_SET'] = fs_name
            timefiltered_dataframe['LEADTIME_MIN'] = counter
            timefiltered_dataframe['HAS_SEVIRI'] = has_seviri

            dataframe_result = pd.concat([dataframe_result, timefiltered_dataframe], ignore_index=True)

            # Compute standard regression metrics for CMV forecast
            step_metrics = regression_results(timefiltered_dataframe["SPECIFIC_YIELD"], timefiltered_dataframe["SPECIFIC_YIELD_PRED_CMV"])
            step_metrics["SCENARIO"] = scenario_name
            step_metrics["FEATURE_SET"] = fs_name
            step_metrics["LEADTIME_MIN"] = counter
            step_metrics["TIMESTAMP"] = date_prediction
            step_metrics["HAS_SEVIRI"] = has_seviri

            # Compute reference metrics and error component breakdown
            if has_seviri:
                step_metrics_org = regression_results(timefiltered_dataframe["SPECIFIC_YIELD"], timefiltered_dataframe["SPECIFIC_YIELD_PRED_ORG"])
                step_metrics["r2_ORG"] = step_metrics_org.get("r2", np.nan)
                step_metrics["RMSE_ORG"] = step_metrics_org.get("RMSE", np.nan)
                step_metrics["MAE_ORG"] = step_metrics_org.get("MAE", np.nan)
                step_metrics["Bias_ORG"] = step_metrics_org.get("Bias", np.nan)
                step_metrics["RMSE_CMV_DIFF"] = np.sqrt(metrics.mean_squared_error(timefiltered_dataframe["SPECIFIC_YIELD_PRED_ORG"], timefiltered_dataframe["SPECIFIC_YIELD_PRED_CMV"]))
                step_metrics["MAE_CMV_DIFF"] = metrics.mean_absolute_error(timefiltered_dataframe["SPECIFIC_YIELD_PRED_ORG"], timefiltered_dataframe["SPECIFIC_YIELD_PRED_CMV"])
            else:
                step_metrics["r2_ORG"] = step_metrics.get("r2", np.nan)
                step_metrics["RMSE_ORG"] = step_metrics.get("RMSE", np.nan)
                step_metrics["MAE_ORG"] = step_metrics.get("MAE", np.nan)
                step_metrics["Bias_ORG"] = step_metrics.get("Bias", np.nan)
                step_metrics["RMSE_CMV_DIFF"] = 0.0
                step_metrics["MAE_CMV_DIFF"] = 0.0

            metrics_results.append(step_metrics)
            counter += forecast_step

        total_steps = (forecast_range // forecast_step) + 1
        if skipped_steps == total_steps:
            print(f"!!! WARNING: '{scenario_name}' / {fs_name} has ZERO usable timesteps - check CSV for {date_predict.date()}")

        if not dataframe_result.empty:
            all_results.append(dataframe_result)

# Combine prediction results across scenarios and feature sets
df_all = pd.concat(all_results, ignore_index=True)
print(df_all.shape)
df_all.head()

df_metrics = pd.DataFrame(metrics_results)

# Build summary rows averaged across leadtimes
id_cols = ["SCENARIO", "FEATURE_SET", "LEADTIME_MIN", "TIMESTAMP", "HAS_SEVIRI"]
metric_cols = [c for c in df_metrics.columns if c not in id_cols]

# Aggregate numeric metrics per scenario and feature set
overall_rows = (
    df_metrics.groupby(["SCENARIO", "FEATURE_SET"])[metric_cols]
    .mean()
    .reset_index()
)
overall_rows["LEADTIME_MIN"] = "ALL"
overall_rows["TIMESTAMP"] = pd.NaT
overall_rows["HAS_SEVIRI"] = overall_rows["FEATURE_SET"].apply(
    lambda fs: len([b for b in all_bands if b in FEATURE_SETS[fs]['features']]) > 0
)

# Append summary rows to metrics dataframe
df_metrics = pd.concat([df_metrics, overall_rows], ignore_index=True)
df_metrics = df_metrics[id_cols + metric_cols]

print(df_metrics.shape)
df_metrics.head(15)

# Save result datasets to CSV files
df_all.to_csv(os.path.join(result_path, "S6_LOC_prediction_results_raw.csv"), index=False, sep=";", decimal=",", encoding="utf-8-sig")
df_metrics.to_csv(os.path.join(result_path, "S6_LOC_prediction_metrics_summary.csv"), index=False, sep=";", decimal=",", encoding="utf-8-sig")

actual/predict datetime:             2018-01-29 07:35:11
rounded datetime last image:         2018-01-29 07_30_00
rounded datetime next to last image: 2018-01-29 07_15_00
#### Scenario SC1 29.01.2018 07:00 Clear winter morning with shadows (2018-01-29 07:35:11)
### using model: FS1_TIME
--Metrics:
  explained_variance:  0.0
  R²:  -0.0253
  MSE:  0.0022
  RMSE:  0.0464
  MAAPE:  0.4177
Dataframe SHAPE:  (132,) (132,)
--Metrics:
  explained_variance:  0.0
  R²:  -0.0862
  MSE:  0.0042
  RMSE:  0.0645
  MAAPE:  0.5078
Dataframe SHAPE:  (133,) (133,)
--Metrics:
  explained_variance:  0.0
  R²:  -0.2199
  MSE:  0.0076
  RMSE:  0.0869
  MAAPE:  0.5711
Dataframe SHAPE:  (134,) (134,)
--Metrics:
  explained_variance:  0.0
  R²:  -0.2874
  MSE:  0.011
  RMSE:  0.1047
  MAAPE:  0.6078
Dataframe SHAPE:  (134,) (134,)
--Metrics:
  explained_variance:  0.0
  R²:  -0.4623
  MSE:  0.0159
  RMSE:  0.1263
  MAAPE:  0.5813
Dataframe SHAPE:  (134,) (134,)
--Metrics:
  explained_variance:  0.0
  R²:  -0.

### Diagrams and Results as CSV

In [16]:
# group per scenario / feature-set / leadtime -> sum over all PV feeders
df_grouped = df_all.groupby(["SCENARIO", "FEATURE_SET", "LEADTIME_MIN"])[["SPECIFIC_YIELD", "SPECIFIC_YIELD_PREDICTION"]].sum().reset_index()

fig = px.line(df_grouped, x="LEADTIME_MIN", y="SPECIFIC_YIELD_PREDICTION", color="FEATURE_SET",
               facet_col="SCENARIO", facet_col_wrap=2,
               title="Predicted Specific Yield per Feature-Set and Leadtime (all Scenarios)")

# observed/true values as a dashed black reference line per facet
# loop trough scenarios because px does not support mixing two y-columns easily)
scenarios = df_grouped["SCENARIO"].unique()
for i, scen in enumerate(scenarios):
    obs = df_grouped[(df_grouped["SCENARIO"] == scen) & (df_grouped["FEATURE_SET"] == df_grouped["FEATURE_SET"].iloc[0])]
    fig.add_scatter(x=obs["LEADTIME_MIN"], y=obs["SPECIFIC_YIELD"], mode="lines",
                     name="Observed" if i == 0 else None, showlegend=(i == 0),
                     line=dict(color="black", dash="dash"),
                     row=(i // 2) + 1, col=(i % 2) + 1)

fig.update_yaxes(title_text="Specific Yield")
fig.update_xaxes(title_text="Leadtime (min)")
fig.show()
fig.write_html(result_path + "/S6_LOC_prediction_timeseries_all_scenarios.html")  # keep interactive version as result

# Diagram: for research question TF2 - error vs leadtime per feature-set

def calc_rmse(group):
    return np.sqrt(np.mean((group["SPECIFIC_YIELD"] - group["SPECIFIC_YIELD_PREDICTION"])**2))

rmse_per_leadtime = df_all.groupby(["FEATURE_SET", "LEADTIME_MIN"]).apply(calc_rmse).reset_index(name="RMSE")

fig2 = px.line(rmse_per_leadtime, x="LEADTIME_MIN", y="RMSE", color="FEATURE_SET",
                title="RMSE of Specific Yield Prediction over Forecast Leadtime (TF2)",
                markers=True)
fig2.update_xaxes(title_text="Leadtime (min)")
fig2.update_yaxes(title_text="RMSE")
fig2.show()
fig2.write_html(result_path + "/S6_LOC_prediction_timeseries_error_vs_leadtime_per_feature-set.html")  # keep interactive version as result

# save the big result dataframe for further analysis
df_all.to_csv(result_path + "/S6_LOC_prediction_results_all_scenarios_featuresets.csv", index=False, sep=";", decimal=",", encoding="utf-8-sig")


plot forecast curves - measured (Ist) vs all feature-sets, aggregated per scenario

In [17]:
# build one aggregated dataframe: mean SPECIFIC_YIELD / PREDICTION per scenario x featureset x leadtime
plot_df = df_all.groupby(["SCENARIO", "FEATURE_SET", "LEADTIME_MIN"]).agg(
    SPECIFIC_YIELD=("SPECIFIC_YIELD", "mean"),
    SPECIFIC_YIELD_PREDICTION=("SPECIFIC_YIELD_PREDICTION", "mean")
).reset_index()

# melt predictions into long-format so every feature-set becomes its own colored line
pred_long = plot_df[["SCENARIO", "FEATURE_SET", "LEADTIME_MIN", "SPECIFIC_YIELD_PREDICTION"]].copy()
pred_long.rename(columns={"SPECIFIC_YIELD_PREDICTION": "VALUE"}, inplace=True)
pred_long["SERIES"] = pred_long["FEATURE_SET"]  # color-key = feature-set name

# measured values are identical for every feature-set at a given scenario/leadtime -> just take FS1 rows
ist_long = plot_df[plot_df["FEATURE_SET"] == "FS1_BASELINE"][["SCENARIO", "LEADTIME_MIN", "SPECIFIC_YIELD"]].copy()
ist_long.rename(columns={"SPECIFIC_YIELD": "VALUE"}, inplace=True)
ist_long["SERIES"] = "IST (measured)"

# stack together - Ist as separate "series" so it gets its own line/color
plot_long = pd.concat([pred_long[["SCENARIO", "LEADTIME_MIN", "SERIES", "VALUE"]],
                        ist_long[["SCENARIO", "LEADTIME_MIN", "SERIES", "VALUE"]]], ignore_index=True)

fig1 = px.line(plot_long, x="LEADTIME_MIN", y="VALUE", color="SERIES",
                facet_col="SCENARIO", facet_col_wrap=2,
                title="SPECIFIC_YIELD forecast vs measured - all feature-sets per scenario",
                labels={"LEADTIME_MIN": "Leadtime [min]", "VALUE": "SPECIFIC_YIELD [kWh/kWp]"})

# make the "Ist" line stand out - black, dashed, thicker line width
for trace in fig1.data:
    if trace.name == "IST (measured)":
        trace.line.color = "black"
        trace.line.dash = "dash"
        trace.line.width = 3

fig1.update_layout(legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="center", x=0.5))
#fig1.write_image(os.path.join(result_path, "forecast_vs_measured_allFS.png"))
fig1.show()
fig1.write_html(result_path + "/S6_LOC_prediction_forecast_vs_measured_allFS.html")  # keep interactive version as result

Heatmaps R2, RMSE, MAAPE and BIAS - for each Scenario and Feature-Set

In [18]:
metrics = ["r2", "RMSE", "MAAPE", "Bias"]

# Filter only the aggregated "ALL" rows from array
all_df = df_metrics[df_metrics["LEADTIME_MIN"] == "ALL"].copy()

for metric in metrics:
    # 1. Pivot into matrix form for the heatmap
    pivot_matrix = all_df.pivot(index="SCENARIO", columns="FEATURE_SET", values=metric)
    print(f"Shape for {metric.upper()}: {pivot_matrix.shape}")
    
    # Use full feature-set names (no shortening)
    x_labels = [str(col) for col in pivot_matrix.columns]
    y_labels = [str(idx) for idx in pivot_matrix.index]

    # 2. Handle clipping / styling depending on the metric
    if metric == "r2":
        z_clipped = pivot_matrix.clip(-2, 1).values
        z_text = pivot_matrix.round(2).astype(str).values
        colorscale = "RdYlGn"
        zmid, zmin, zmax = 0, -2, 1
        colorbar_title = "R2"
        title_text = "Heatmap für Metrik R² je Szenario und Feature-Set über den Prognosezeitraum 0 - 180 Minuten"
        tickvals = [-2, -1, 0, 1]
    elif metric == "RMSE":
        # lower RMSE is better -> no hard clipping needed, adjust if outliers dominate the scale
        z_clipped = pivot_matrix.values
        z_text = pivot_matrix.round(2).astype(str).values
        colorscale = "RdYlGn_r"  # _r inverts scale because smaller RMSE = better
        zmid, zmin, zmax = None, None, None
        colorbar_title = "RMSE"
        title_text = "Heatmap für Metrik RMSE je Szenario und Feature-Set über den Prognosezeitraum 0 - 180 Minuten"
        tickvals = None
    elif metric == "MAAPE":
        # smaller MAAPE is better
        z_clipped = pivot_matrix.values
        z_text = pivot_matrix.round(2).astype(str).values
        colorscale = "RdYlGn_r"
        zmid, zmin, zmax = None, None, None
        colorbar_title = "MAAPE"
        title_text = "Heatmap für Metrik MAAPE je Szenario und Feature-Set über den Prognosezeitraum 0 - 180 Minuten"
        tickvals = None
    else:  # bias
        # bias close to 0 is good - positive = model underestimates, negative = model overestimates
        # symmetric clipping around 0, use max absolute value across the matrix (ignore extreme scenario outliers)
        bias_limit = min(pivot_matrix.abs().quantile(0.9).max(), 0.5)  # cap so a single crazy scenario doesn't flatten colors
        z_clipped = pivot_matrix.clip(-bias_limit, bias_limit).values
        z_text = pivot_matrix.round(3).astype(str).values  # more decimals, bias values are usually small
        colorscale = "RdBu"  # diverging: blue = underestimate, red = overestimate
        zmid, zmin, zmax = 0, -bias_limit, bias_limit
        colorbar_title = "Bias"
        title_text = "Heatmap für Metrik Bias je Szenario und Feature-Set über den Prognosezeitraum 0 - 180 Minuten"
        tickvals = None

    # 3. Create Figure
    fig = go.Figure(data=go.Heatmap(
        z=z_clipped,
        x=x_labels,
        y=y_labels,
        text=z_text,
        texttemplate="%{text}",
        textfont=dict(size=12),
        colorscale=colorscale,
        zmid=zmid,
        zmin=zmin,
        zmax=zmax,
        colorbar=dict(title=colorbar_title, tickvals=tickvals)
    ))

    fig.update_layout(
        title=title_text,
        height=650,
        width=1200,  # a bit wider because of the full feature-set names
        xaxis=dict(tickangle=-45)  # tilt long names for better readability
    )
    fig.update_xaxes(title_text="Feature-Set")
    fig.update_yaxes(title_text="Szenario", automargin=True)

    # 4. Save and show
    file_name = f"/S6_LOC_{metric.lower()}_heatmap_scenario_featureset.png"
    fig.write_image(result_path + file_name, scale=3)
    fig.show()

all_df


Shape for R2: (8, 9)


Shape for RMSE: (8, 9)


Shape for MAAPE: (8, 9)


Shape for BIAS: (8, 9)


,SCENARIO,FEATURE_SET,LEADTIME_MIN,TIMESTAMP,HAS_SEVIRI,N Samples,r2,explained Variance,MAE,MSE,...,median AE,MAPE,Bias,MAAPE,r2_ORG,RMSE_ORG,MAE_ORG,Bias_ORG,RMSE_CMV_DIFF,MAE_CMV_DIFF
936,SC1 29.01.2018 07:00 Clear winter morning with...,FS1_TIME,ALL,NaT,False,133.769231,-1.455563,0.000000,0.137836,0.036708,...,0.127013,0.776289,0.129255,0.532230,-1.455563,0.173635,0.137836,0.129255,0.000000,0.000000
937,SC1 29.01.2018 07:00 Clear winter morning with...,FS2_GIS_TIME,ALL,NaT,False,133.769231,-0.967882,0.233665,0.126839,0.030088,...,0.119124,0.969828,0.114345,0.565791,-0.967882,0.155328,0.126839,0.114345,0.000000,0.000000
938,SC1 29.01.2018 07:00 Clear winter morning with...,FS3_GIS_SENSOR_TIME,ALL,NaT,False,133.769231,-3.533886,0.169727,0.207275,0.072194,...,0.203066,0.980524,0.207007,0.723628,-3.533886,0.234395,0.207275,0.207007,0.000000,0.000000
939,SC1 29.01.2018 07:00 Clear winter morning with...,FS4_SEV4_TIME,ALL,NaT,True,133.769231,-0.890207,-0.034333,0.117079,0.027378,...,0.110596,1.240021,0.094917,0.558516,-0.171799,0.118479,0.081647,0.035993,0.063659,0.059698
940,SC1 29.01.2018 07:00 Clear winter morning with...,FS5_SEV4_GIS_TIME,ALL,NaT,True,133.769231,0.313020,0.434465,0.071593,0.009604,...,0.061626,1.111154,0.030400,0.463333,0.384224,0.086746,0.065807,0.008462,0.037289,0.031815
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1003,SC8 16.12.2018 11:00 Winter day with snow and ...,FS5_SEV4_GIS_TIME,ALL,NaT,True,60.153846,-25.713350,-5.102457,0.047833,0.003334,...,0.045910,31.968197,-0.044390,1.315811,-42.113998,0.070587,0.064450,-0.063545,0.028506,0.024624
1004,SC8 16.12.2018 11:00 Winter day with snow and ...,FS6_SEV4_GIS_SENSOR_TIME,ALL,NaT,True,60.153846,-10.821025,-2.925708,0.031952,0.001473,...,0.027967,16.877880,-0.030255,1.282158,-16.869190,0.046159,0.039082,-0.038237,0.013375,0.009358
1005,SC8 16.12.2018 11:00 Winter day with snow and ...,FS7_SEV12_TIME,ALL,NaT,True,60.153846,-25.729163,-1.931145,0.052307,0.003340,...,0.054968,39.164958,-0.051746,1.337646,-28.859169,0.059179,0.056527,-0.056419,0.021765,0.017393
1006,SC8 16.12.2018 11:00 Winter day with snow and ...,FS8_SEV12_GIS_TIME,ALL,NaT,True,60.153846,-15.094905,-3.240005,0.037471,0.002117,...,0.034757,23.431360,-0.031506,1.273541,-29.983937,0.060909,0.055252,-0.054465,0.031934,0.027832


In [19]:
# Compute error decomposition per leadtime and featureset
df_seviri = df_all[df_all["HAS_SEVIRI"] == True].copy()

# Calculate squared errors per row
df_seviri["SE_TOTAL"] = (df_seviri["SPECIFIC_YIELD_PRED_CMV"] - df_seviri["SPECIFIC_YIELD"]) ** 2
df_seviri["SE_REG"] = (df_seviri["SPECIFIC_YIELD_PRED_ORG"] - df_seviri["SPECIFIC_YIELD"]) ** 2
df_seviri["SE_CMV"] = (df_seviri["SPECIFIC_YIELD_PRED_CMV"] - df_seviri["SPECIFIC_YIELD_PRED_ORG"]) ** 2

# Aggregate error components across all scenarios and feeders
decomp = df_seviri.groupby(["FEATURE_SET", "LEADTIME_MIN"]).agg(
    MSE_TOTAL=("SE_TOTAL", "mean"),
    MSE_REG=("SE_REG", "mean"),
    MSE_CMV=("SE_CMV", "mean"),
    MAE_TOTAL=("ERROR_TOTAL", lambda x: np.mean(np.abs(x))),
    MAE_REG=("ERROR_REGRESSION", lambda x: np.mean(np.abs(x))),
    MAE_CMV=("ERROR_CMV", lambda x: np.mean(np.abs(x)))
).reset_index()

# Calculate RMSE and relative percentage shares
decomp["RMSE_TOTAL"] = np.sqrt(decomp["MSE_TOTAL"])
decomp["RMSE_REG"] = np.sqrt(decomp["MSE_REG"])
decomp["RMSE_CMV"] = np.sqrt(decomp["MSE_CMV"])
decomp["DELTA_RMSE"] = decomp["RMSE_TOTAL"] - decomp["RMSE_REG"]

sum_mse = decomp["MSE_REG"] + decomp["MSE_CMV"]
decomp["SHARE_REG_PCT"] = (decomp["MSE_REG"] / sum_mse) * 100
decomp["SHARE_CMV_PCT"] = (decomp["MSE_CMV"] / sum_mse) * 100

# Save decomposed metrics table to CSV
decomp_path = os.path.join(result_path, "S6_LOC_error_decomposition_by_leadtime.csv")
decomp.to_csv(decomp_path, index=False, sep=";", decimal=",", encoding="utf-8-sig")
print("Saved decomposition table to:", decomp_path)


# Diagram: Relative Error Share over Leadtime
fig_share = go.Figure()

# Average across feature sets for global view
global_decomp = decomp.groupby("LEADTIME_MIN")[["SHARE_REG_PCT", "SHARE_CMV_PCT"]].mean().reset_index()

fig_share.add_trace(go.Bar(
    x=global_decomp["LEADTIME_MIN"],
    y=global_decomp["SHARE_REG_PCT"],
    name="Regressionsanalyse (CatBoost)",
    marker_color="#2b5c8f",
    text=global_decomp["SHARE_REG_PCT"].round(1).astype(str) + "%",
    textposition="inside"
))

fig_share.add_trace(go.Bar(
    x=global_decomp["LEADTIME_MIN"],
    y=global_decomp["SHARE_CMV_PCT"],
    name="Wolkenbewegungsmodell (CMV / Optical Flow)",
    marker_color="#e26d5c",
    text=global_decomp["SHARE_CMV_PCT"].round(1).astype(str) + "%",
    textposition="inside"
))

fig_share.update_layout(
    barmode="stack",
    title="Fehleranteile am Gesamtfehler: Regressionsanalyse vs. CMV-Extrapolation (15-180 min)",
    xaxis=dict(title="Prognosehorizont (Leadtime) [min]", tickmode="linear", dtick=15),
    yaxis=dict(title="Relativer Fehleranteil [%]", range=[0, 100]),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5),
    template="plotly_white",
    height=550,
    width=950
)
fig_share.show()
fig_share.write_image(os.path.join(result_path, "S6_LOC_error_share_stacked_bar.png"), scale=3)
fig_share.write_html(os.path.join(result_path, "S6_LOC_error_share_stacked_bar.html"))


# Diagram: RMSE Progression and Delta over Leadtime
fig_rmse = go.Figure()

# Plot RMSE Total vs RMSE Regression Oracle
fig_rmse.add_trace(go.Scatter(
    x=global_decomp["LEADTIME_MIN"],
    y=decomp.groupby("LEADTIME_MIN")["RMSE_TOTAL"].mean(),
    mode="lines+markers",
    name="RMSE Hybrid (CMV + CatBoost)",
    line=dict(color="#d90429", width=3)
))

fig_rmse.add_trace(go.Scatter(
    x=global_decomp["LEADTIME_MIN"],
    y=decomp.groupby("LEADTIME_MIN")["RMSE_REG"].mean(),
    mode="lines+markers",
    name="RMSE Regression CatBoost mit gemessenen SEVIRI-Daten",
    line=dict(color="#2b2d42", width=2, dash="dash")
))

fig_rmse.add_trace(go.Scatter(
    x=global_decomp["LEADTIME_MIN"],
    y=decomp.groupby("LEADTIME_MIN")["DELTA_RMSE"].mean(),
    mode="lines+markers",
    name="Δ RMSE (Verlust durch CMV-Ungenauigkeit)",
    line=dict(color="#8d99ae", width=2, dash="dot")
))

fig_rmse.update_layout(
    title="Fehlerdivergenz über den Prognosezeitraum 15 bis 180 Minuten",
    xaxis=dict(title="Prognosehorizont [min]", tickmode="linear", dtick=15),
    yaxis=dict(title="RMSE [kWh/kWp]"),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5),
    template="plotly_white",
    height=550,
    width=950
)
fig_rmse.show()
fig_rmse.write_image(os.path.join(result_path, "S6_LOC_rmse_progression_leadtime.png"), scale=3)
fig_rmse.write_html(os.path.join(result_path, "S6_LOC_rmse_progression_leadtime.html"))

Saved decomposition table to: .\results\S6_LOC_error_decomposition_by_leadtime.csv


In [20]:
# Error Decomposition with scenario breakdown
df_seviri = df_all[df_all["HAS_SEVIRI"] == True].copy()

# Calculate squared error components
df_seviri["SE_TOTAL"] = (df_seviri["SPECIFIC_YIELD_PRED_CMV"] - df_seviri["SPECIFIC_YIELD"]) ** 2
df_seviri["SE_REG"] = (df_seviri["SPECIFIC_YIELD_PRED_ORG"] - df_seviri["SPECIFIC_YIELD"]) ** 2
df_seviri["SE_CMV"] = (df_seviri["SPECIFIC_YIELD_PRED_CMV"] - df_seviri["SPECIFIC_YIELD_PRED_ORG"]) ** 2

# Aggregate by SCENARIO, FEATURE_SET and LEADTIME_MIN
decomp_scen = df_seviri.groupby(["SCENARIO", "FEATURE_SET", "LEADTIME_MIN"]).agg(
    MSE_TOTAL=("SE_TOTAL", "mean"),
    MSE_REG=("SE_REG", "mean"),
    MSE_CMV=("SE_CMV", "mean"),
    MAE_TOTAL=("ERROR_TOTAL", lambda x: np.mean(np.abs(x))),
    MAE_REG=("ERROR_REGRESSION", lambda x: np.mean(np.abs(x))),
    MAE_CMV=("ERROR_CMV", lambda x: np.mean(np.abs(x)))
).reset_index()

# Calculate RMSE and percentage error shares
decomp_scen["RMSE_TOTAL"] = np.sqrt(decomp_scen["MSE_TOTAL"])
decomp_scen["RMSE_REG"] = np.sqrt(decomp_scen["MSE_REG"])
decomp_scen["RMSE_CMV"] = np.sqrt(decomp_scen["MSE_CMV"])
decomp_scen["DELTA_RMSE"] = decomp_scen["RMSE_TOTAL"] - decomp_scen["RMSE_REG"]

sum_mse = decomp_scen["MSE_REG"] + decomp_scen["MSE_CMV"]
# Avoid division by zero for leadtime 0
decomp_scen["SHARE_REG_PCT"] = np.where(sum_mse > 0, (decomp_scen["MSE_REG"] / sum_mse) * 100, 100.0)
decomp_scen["SHARE_CMV_PCT"] = np.where(sum_mse > 0, (decomp_scen["MSE_CMV"] / sum_mse) * 100, 0.0)

# Export complete table with SCENARIO column
scen_csv_path = os.path.join(result_path, "S6_LOC_error_decomposition_by_scenario_leadtime.csv")
decomp_scen.to_csv(scen_csv_path, index=False, sep=";", decimal=",", encoding="utf-8-sig")
print("Saved scenario decomposition table to:", scen_csv_path)


# Heatmaps: CMV Error Share per Scenario

# Plot heatmap for the standard satellite+GIS model (FS8_SEV12_GIS_TIME)
fs_target = "FS8_SEV12_GIS_TIME"
sub_fs = decomp_scen[decomp_scen["FEATURE_SET"] == fs_target].copy()
pivot_scen = sub_fs.pivot(index="SCENARIO", columns="LEADTIME_MIN", values="SHARE_CMV_PCT")

fig_scen_hm = go.Figure(data=go.Heatmap(
    z=pivot_scen.values,
    x=[f"{int(c)}" for c in pivot_scen.columns],
    y=list(pivot_scen.index),
    text=pivot_scen.round(1).astype(str) + "%",
    texttemplate="%{text}",
    textfont={"size": 11},
    colorscale="YlOrRd",
    colorbar=dict(title="CMV-Fehler [%]")
))

fig_scen_hm.update_layout(
    title=f"CMV-Fehleranteil [%] nach Wetterszenario und Leadtime ({fs_target})",
    xaxis=dict(title="Prognosehorizont (Leadtime) [min]"),
    yaxis=dict(title="Szenario", automargin=True),
    template="plotly_white",
    height=600,
    width=1400
)
fig_scen_hm.show()
fig_scen_hm.write_image(os.path.join(result_path, "S6_LOC_heatmap_cmv_share_by_scenario.png"), scale=3)
fig_scen_hm.write_html(os.path.join(result_path, "S6_LOC_heatmap_cmv_share_by_scenario.html"))

Saved scenario decomposition table to: .\results\S6_LOC_error_decomposition_by_scenario_leadtime.csv


In [21]:
print('Describe Dataframe dataframe_result:')
print(df_all.head(5))
print(df_all.describe())
print(df_all.shape)
df_all.info()
df_all

Describe Dataframe dataframe_result:
   Unnamed: 0  NEI_ID  HAUS_ID                      ZAEHLPUNKT  \
0      213722    1003    61490  AT0060000693400000000000218628   
1     1631882    3325    41994  AT0060000682200000000099000000   
2     1312796    3780    59329  DE0003908817900000000007000019   
3     2199146    6954   118839  AT0060000690000000000099000000   
4     2465051    9420   135515  DE0003908813800000000007000037   

            TIMESTAMP  RECHTSWERT_GK  HOCHWERT_GK  LON_WGS84  LAT_WGS84  \
0 2018-01-29 07:30:00     -35091.785    264074.03   9.867123  47.514404   
1 2018-01-29 07:30:00     -50684.906    231261.95   9.663879  47.218260   
2 2018-01-29 07:30:00     -29557.537    269269.30   9.940242  47.561405   
3 2018-01-29 07:30:00     -44779.742    261206.70   9.738803  47.488014   
4 2018-01-29 07:30:00     -41521.910    273063.20   9.780906  47.594868   

   HEIGHT  ...  WV_062_ORG  IR_108_ORG VIS008_ORG  IR_016_ORG  WV_073_ORG  \
0  554.51  ...         NaN         NaN

,Unnamed: 0,NEI_ID,HAUS_ID,ZAEHLPUNKT,TIMESTAMP,RECHTSWERT_GK,HOCHWERT_GK,LON_WGS84,LAT_WGS84,HEIGHT,...,WV_062_ORG,IR_108_ORG,VIS008_ORG,IR_016_ORG,WV_073_ORG,IR_087_ORG,IR_097_ORG,IR_120_ORG,IR_134_ORG,HRV_ORG
0,213722,1003,61490,AT0060000693400000000000218628,2018-01-29 07:30:00,-35091.785,264074.03,9.867123,47.514404,554.51,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1631882,3325,41994,AT0060000682200000000099000000,2018-01-29 07:30:00,-50684.906,231261.95,9.663879,47.218260,477.76,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1312796,3780,59329,DE0003908817900000000007000019,2018-01-29 07:30:00,-29557.537,269269.30,9.940242,47.561405,846.43,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2199146,6954,118839,AT0060000690000000000099000000,2018-01-29 07:30:00,-44779.742,261206.70,9.738803,47.488014,414.53,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2465051,9420,135515,DE0003908813800000000007000037,2018-01-29 07:30:00,-41521.910,273063.20,9.780906,47.594868,462.32,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118732,1151737,21639,82899,AT0060000671900000000099000004,2018-12-16 14:00:00,-46656.797,228059.69,9.717390,47.189760,505.94,...,2.387298,52.899200,1.975838,1.162055,10.350687,28.010492,24.846659,64.247925,60.678654,2.149548
118733,52663,21771,97835,AT0060000683300000000099000004,2018-12-16 14:00:00,-52990.086,241106.75,9.632276,47.306620,446.82,...,2.329071,51.874023,3.645560,2.603003,10.041712,27.630259,24.638737,63.580994,59.417797,3.911474
118734,1027648,21932,151884,AT0060000697100000000099000004,2018-12-16 14:00:00,-48135.700,261030.73,9.694293,47.486195,399.45,...,2.528706,58.230133,1.669722,0.952885,11.277615,31.432589,26.094189,70.917260,64.776436,2.466695
118735,159025,22039,48299,AT0060000685000000000099000004,2018-12-16 14:00:00,-44726.477,255165.17,9.740125,47.433680,412.84,...,2.495433,56.384810,1.419264,0.743715,11.007261,30.165148,25.470423,68.916460,63.357975,1.550494
